<a href="https://colab.research.google.com/github/llv1005/Ing_Gest_Poyectos/blob/main/Copia_de_sgic_sesion1_langchain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sesión 1 · Primer contacto con LangChain

**Sistemas de Gestión de la Información Clínica** · Grado en Ingeniería de la Salud

En la presentación hemos visto qué hace un modelo de lenguaje: recibe texto y predice, token a token, el texto que viene después. En este notebook vamos a usar un LLM a través de **LangChain** para resolver un problema muy habitual en información clínica: **convertir texto clínico libre en información estructurada y útil**.

El código ya está escrito. Tu trabajo consiste en **ejecutarlo, modificar los prompts y observar qué cambia**. Las celdas marcadas con ✏️ son las que debes tocar.

| Bloque | Qué aprenderás |
|---|---|
| 1. Llamada básica | Mensajes de sistema y de usuario; qué devuelve el modelo |
| 2. Plantillas de prompts | Reutilizar un mismo prompt con variables |
| 3. Temperatura | Por qué el mismo prompt no siempre da la misma respuesta |
| 4. Salida estructurada | Extraer diagnósticos, negaciones y antecedentes familiares con un esquema |
| 5. Cadenas y lotes | Procesar 10 informes y **medir** cuántos diagnósticos acierta el modelo |

Lo que programes aquí reaparecerá en las sesiones 2 y 3 como componentes visuales de Langflow.

> ⚠️ **Datos.** Todos los informes de este notebook son **sintéticos**. Nunca envíes datos reales de pacientes a un servicio externo: los datos de salud son una categoría especial en el RGPD.

## 0. Preparación del entorno

Instalamos las librerías. Si trabajas en Google Colab o en un entorno nuevo, ejecuta la celda siguiente (solo hace falta una vez).

### Entorno local (pc)

In [ ]:
%pip install -q -U langchain langchain-openai langchain-google-genai pydantic pandas

### Gggole Colab

In [ ]:
%pip install -q -U langchain langchain-openai langchain-google-genai

### Elegir proveedor y clave de API

Puedes trabajar con dos proveedores:

- **OpenAI**, con la clave proporcionada en clase (tiene un límite de gasto). Modelo recomendado: `gpt-4.1-nano`, un modelo pequeño y barato **sin razonamiento**.
- **Google**, con tu propia cuenta en el plan gratuito de Google AI Studio (<https://aistudio.google.com/apikey>). El plan gratuito limita el número de peticiones por minuto, así que algunas celdas pueden ir más lentas.

✏️ Cambia `PROVEEDOR` si vas a usar Google. La clave se pide de forma oculta y **no se guarda en el notebook**: no la escribas nunca directamente en una celda. En Colab las claves se gurdan en secretos 🔑 (barra lateral)

In [ ]:
PROVEEDOR = "openai"        # "openai" o "google"
MODELOS = {
    "openai": "gpt-4.1-nano",
    "google": "gemini-3.5-flash-lite",   # si no está disponible, consulta los modelos del plan gratuito en AI Studio
}
VARIABLE_CLAVE = {"openai": "OPENAI_API_KEY", "google": "GOOGLE_API_KEY"}

### Colab
Debes tener guardada la clave en secretos 🔑 (barra lateral)

In [ ]:
import os
from google.colab import userdata
os.environ[VARIABLE_CLAVE[PROVEEDOR]] = userdata.get(VARIABLE_CLAVE[PROVEEDOR])
print("Proveedor:", PROVEEDOR, "· Modelo:", MODELOS[PROVEEDOR])

Proveedor: openai · Modelo: gpt-4.1-nano


### En local
Debes tener guardada la clave localmente

In [ ]:
import os
from getpass import getpass
if not os.environ.get(VARIABLE_CLAVE[PROVEEDOR]):
    os.environ[VARIABLE_CLAVE[PROVEEDOR]] = getpass(f"Introduce tu {VARIABLE_CLAVE[PROVEEDOR]}: ")

print("Proveedor:", PROVEEDOR, "· Modelo:", MODELOS[PROVEEDOR])

Proveedor: openai · Modelo: gpt-4.1-nano


### Crear una instancia de un modelo

In [ ]:
from langchain.chat_models import init_chat_model

def crear_modelo(temperatura=0.0):
    # Devuelve un modelo de chat del proveedor elegido con la temperatura indicada.
    return init_chat_model(
        MODELOS[PROVEEDOR],
        model_provider="openai" if PROVEEDOR == "openai" else "google_genai",
        temperature=temperatura,
    )

llm = crear_modelo(temperatura=0)

### Los datos: 10 informes clínicos breves

Trabajaremos con diez fragmentos de informes clínicos sintéticos. Cada uno lleva una **gold standard** (etiqueta de referencia validada por expertos) con los diagnósticos que menciona y dos atributos:

- `negado`: el texto dice que el paciente **no** tiene ese diagnóstico ("se descarta neumonía").
- `antecedente_familiar`: el diagnóstico lo tiene **un familiar**, no el paciente ("madre con cáncer de mama").

Estos dos atributos son la causa más frecuente de errores al extraer información de texto clínico: un sistema que solo busca la palabra "neumonía" diría que el paciente tiene neumonía. La anotación la usaremos en el bloque 5 para evaluar el modelo.

In [ ]:
# Datos sintéticos
INFORMES = [
    {"id": 1,
     "texto": "Varón de 67 años con hipertensión arterial y diabetes mellitus tipo 2 que acude por dolor torácico. "
              "ECG sin alteraciones y troponinas normales; se descarta infarto agudo de miocardio. "
              "Padre fallecido por cáncer de colon.",
     "gold": [
         {"diagnostico": "hipertensión arterial", "negado": False, "antecedente_familiar": False},
         {"diagnostico": "diabetes mellitus tipo 2", "negado": False, "antecedente_familiar": False},
         {"diagnostico": "infarto agudo de miocardio", "negado": True, "antecedente_familiar": False},
         {"diagnostico": "cáncer de colon", "negado": False, "antecedente_familiar": True},
     ]},
    {"id": 2,
     "texto": "Mujer de 45 años con antecedentes personales de asma bronquial. Consulta por tos y fiebre de tres días. "
              "La radiografía de tórax no muestra condensaciones, por lo que se descarta neumonía. "
              "Juicio clínico: bronquitis aguda.",
     "gold": [
         {"diagnostico": "asma bronquial", "negado": False, "antecedente_familiar": False},
         {"diagnostico": "neumonía", "negado": True, "antecedente_familiar": False},
         {"diagnostico": "bronquitis aguda", "negado": False, "antecedente_familiar": False},
     ]},
    {"id": 3,
     "texto": "Mujer de 52 años con fibroadenoma de mama izquierda diagnosticado por ecografía hace dos años, "
              "estable en el control actual. Madre y tía materna con cáncer de mama.",
     "gold": [
         {"diagnostico": "fibroadenoma de mama", "negado": False, "antecedente_familiar": False},
         {"diagnostico": "cáncer de mama", "negado": False, "antecedente_familiar": True},
     ]},
    {"id": 4,
     "texto": "Varón de 30 años sin antecedentes de interés. Acude a urgencias por dolor en fosa ilíaca derecha y diarrea. "
              "Ecografía abdominal normal; se descarta apendicitis aguda. Diagnóstico al alta: gastroenteritis aguda.",
     "gold": [
         {"diagnostico": "apendicitis aguda", "negado": True, "antecedente_familiar": False},
         {"diagnostico": "gastroenteritis aguda", "negado": False, "antecedente_familiar": False},
     ]},
    {"id": 5,
     "texto": "Mujer de 78 años con fibrilación auricular en tratamiento anticoagulante e insuficiencia cardiaca crónica. "
              "Ingresa por descompensación de su insuficiencia cardiaca. Hermano con enfermedad de Alzheimer.",
     "gold": [
         {"diagnostico": "fibrilación auricular", "negado": False, "antecedente_familiar": False},
         {"diagnostico": "insuficiencia cardiaca crónica", "negado": False, "antecedente_familiar": False},
         {"diagnostico": "enfermedad de Alzheimer", "negado": False, "antecedente_familiar": True},
     ]},
    {"id": 6,
     "texto": "Niño de 8 años con dermatitis atópica desde la lactancia. Su padre tiene psoriasis. "
              "En la exploración no presenta signos de infección cutánea.",
     "gold": [
         {"diagnostico": "dermatitis atópica", "negado": False, "antecedente_familiar": False},
         {"diagnostico": "psoriasis", "negado": False, "antecedente_familiar": True},
         {"diagnostico": "infección cutánea", "negado": True, "antecedente_familiar": False},
     ]},
    {"id": 7,
     "texto": "Varón de 59 años con EPOC y obesidad. Se solicita polisomnografía por sospecha de síndrome de apnea "
              "obstructiva del sueño. Sin antecedentes familiares de enfermedad cardiovascular.",
     "gold": [
         {"diagnostico": "EPOC", "negado": False, "antecedente_familiar": False},
         {"diagnostico": "obesidad", "negado": False, "antecedente_familiar": False},
         {"diagnostico": "síndrome de apnea obstructiva del sueño", "negado": False, "antecedente_familiar": False},
         {"diagnostico": "enfermedad cardiovascular", "negado": True, "antecedente_familiar": True},
     ]},
    {"id": 8,
     "texto": "Mujer de 34 años, gestante de 20 semanas. La sobrecarga oral de glucosa es normal, por lo que se descarta "
              "diabetes gestacional. Presenta anemia ferropénica. Madre con hipotiroidismo.",
     "gold": [
         {"diagnostico": "diabetes gestacional", "negado": True, "antecedente_familiar": False},
         {"diagnostico": "anemia ferropénica", "negado": False, "antecedente_familiar": False},
         {"diagnostico": "hipotiroidismo", "negado": False, "antecedente_familiar": True},
     ]},
    {"id": 9,
     "texto": "Varón de 71 años con cardiopatía isquémica e insuficiencia renal crónica estadio 3. "
              "No refiere antecedentes de ictus. Su hermana padece diabetes mellitus tipo 1.",
     "gold": [
         {"diagnostico": "cardiopatía isquémica", "negado": False, "antecedente_familiar": False},
         {"diagnostico": "insuficiencia renal crónica", "negado": False, "antecedente_familiar": False},
         {"diagnostico": "ictus", "negado": True, "antecedente_familiar": False},
         {"diagnostico": "diabetes mellitus tipo 1", "negado": False, "antecedente_familiar": True},
     ]},
    {"id": 10,
     "texto": "Mujer de 25 años que consulta por cefalea intensa. Exploración neurológica normal. Se descarta hemorragia "
              "subaracnoidea mediante TC craneal. Juicio clínico: migraña sin aura. Antecedentes familiares de migraña "
              "por línea materna.",
     "gold": [
         {"diagnostico": "hemorragia subaracnoidea", "negado": True, "antecedente_familiar": False},
         {"diagnostico": "migraña sin aura", "negado": False, "antecedente_familiar": False},
         {"diagnostico": "migraña", "negado": False, "antecedente_familiar": True},
     ]},
]

# Sinónimos aceptados al comparar con la referencia (el modelo puede desarrollar siglas, etc.)
SINONIMOS = {
    "EPOC": ["enfermedad pulmonar obstructiva crónica"],
    "síndrome de apnea obstructiva del sueño": ["apnea obstructiva del sueño", "SAOS", "SAHS", "apnea del sueño"],
    "ictus": ["accidente cerebrovascular", "enfermedad cerebrovascular", "ACV"],
    "enfermedad de Alzheimer": ["Alzheimer"],
    "insuficiencia cardiaca crónica": ["insuficiencia cardíaca crónica"],
}

print(f"{len(INFORMES)} informes, {sum(len(i['gold']) for i in INFORMES)} diagnósticos anotados")

10 informes, 31 diagnósticos anotados


---
## Bloque 1 · Llamada básica al modelo

Un modelo de chat recibe una lista de **mensajes**. Los dos tipos más importantes son:

- **Mensaje de sistema** (`SystemMessage`): instrucciones generales sobre el papel del modelo y cómo debe responder. El usuario final normalmente no lo ve.
- **Mensaje de usuario** (`HumanMessage`): la petición concreta.

El modelo devuelve un `AIMessage`. Entra texto y sale texto.

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage
from IPython.display import display, Markdown

mensajes = [
    SystemMessage(content="Eres un asistente que ayuda a profesionales sanitarios. Responde en español, "
                          "de forma breve y sin añadir información que no aparezca en el texto."),
    HumanMessage(content="Resume en una frase el siguiente informe:\n\n" + INFORMES[0]["texto"]),
]

respuesta = llm.invoke(mensajes)
display(Markdown(respuesta.content))

Varón de 67 años con hipertensión y diabetes que presenta dolor torácico, ECG y troponinas normales, descartando infarto agudo de miocardio.

### Qué hay dentro de la respuesta

Además del texto, la respuesta trae metadatos: qué modelo respondió, por qué terminó de generar y **cuántos tokens** se consumieron en la entrada y en la salida. Los tokens son lo que se factura y lo que ocupa la ventana de contexto.

In [ ]:
print("Tipo de objeto:  ", type(respuesta).__name__)
print("Contenido:       ", respuesta.content[:80], "...")
print("Uso de tokens:   ", respuesta.usage_metadata)
print("Metadatos:       ", {k: v for k, v in respuesta.response_metadata.items() if k in ("model_name", "finish_reason")})

Tipo de objeto:   AIMessage
Contenido:        Varón de 67 años con hipertensión y diabetes que presenta dolor torácico, ECG y  ...
Uso de tokens:    {'input_tokens': 106, 'output_tokens': 36, 'total_tokens': 142, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}
Metadatos:        {'model_name': 'gpt-4.1-nano-2025-04-14', 'finish_reason': 'stop'}


✏️ **Prueba tú.** Modifica el mensaje de sistema de la celda anterior y vuelve a ejecutar:

1. Pide que responda "como si hablara con el paciente, sin tecnicismos".
2. Pide que responda **solo** con una lista de diagnósticos separados por comas.
3. Elimina el mensaje de sistema. ¿Cambia la longitud de la respuesta? ¿Y el número de tokens de salida?

In [ ]:
mensajes = [
    SystemMessage(content="Responde como si hablaras con el paciente, sin tecnicismos."),
    HumanMessage(content="Resume en una frase el siguiente informe:\n\n" + INFORMES[0]["texto"]),
]

respuesta = llm.invoke(mensajes)
display(Markdown(respuesta.content))
print("Tipo de objeto:  ", type(respuesta).__name__)
print("Contenido:       ", respuesta.content[:80], "...")
print("Uso de tokens:   ", respuesta.usage_metadata)
print("Metadatos:       ", {k: v for k, v in respuesta.response_metadata.items() if k in ("model_name", "finish_reason")})

Eres un hombre de 67 años con hipertensión y diabetes que vino por dolor en el pecho, pero las pruebas no muestran un infarto, y tu familia tiene antecedentes de cáncer de colon.

Tipo de objeto:   AIMessage
Contenido:        Eres un hombre de 67 años con hipertensión y diabetes que vino por dolor en el p ...
Uso de tokens:    {'input_tokens': 89, 'output_tokens': 40, 'total_tokens': 129, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}
Metadatos:        {'model_name': 'gpt-4.1-nano-2025-04-14', 'finish_reason': 'stop'}


In [ ]:
mensajes = [
    SystemMessage(content="Responde solo con una lista de diagnósticos separados por comas."),
    HumanMessage(content="Resume en una frase el siguiente informe:\n\n" + INFORMES[0]["texto"]),
]

respuesta = llm.invoke(mensajes)
display(Markdown(respuesta.content))
print("Tipo de objeto:  ", type(respuesta).__name__)
print("Contenido:       ", respuesta.content[:80], "...")
print("Uso de tokens:   ", respuesta.usage_metadata)
print("Metadatos:       ", {k: v for k, v in respuesta.response_metadata.items() if k in ("model_name", "finish_reason")})

Dolor torácico en varón de 67 años con hipertensión y diabetes, sin evidencia de infarto agudo de miocardio, antecedentes familiares de cáncer de colon.

Tipo de objeto:   AIMessage
Contenido:        Dolor torácico en varón de 67 años con hipertensión y diabetes, sin evidencia de ...
Uso de tokens:    {'input_tokens': 89, 'output_tokens': 37, 'total_tokens': 126, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}
Metadatos:        {'model_name': 'gpt-4.1-nano-2025-04-14', 'finish_reason': 'stop'}


In [ ]:
mensajes = [
    HumanMessage(content="Resume en una frase el siguiente informe:\n\n" + INFORMES[0]["texto"]),
]

respuesta = llm.invoke(mensajes)
display(Markdown(respuesta.content))
print("Tipo de objeto:  ", type(respuesta).__name__)
print("Contenido:       ", respuesta.content[:80], "...")
print("Uso de tokens:   ", respuesta.usage_metadata)
print("Metadatos:       ", {k: v for k, v in respuesta.response_metadata.items() if k in ("model_name", "finish_reason")})

Varón de 67 años con hipertensión y diabetes que presenta dolor torácico, ECG y troponinas normales, descartándose infarto agudo de miocardio, con antecedentes familiares de cáncer de colon.

Tipo de objeto:   AIMessage
Contenido:        Varón de 67 años con hipertensión y diabetes que presenta dolor torácico, ECG y  ...
Uso de tokens:    {'input_tokens': 70, 'output_tokens': 44, 'total_tokens': 114, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}
Metadatos:        {'model_name': 'gpt-4.1-nano-2025-04-14', 'finish_reason': 'stop'}


---
## Bloque 2 · Plantillas de prompts

En una aplicación real no escribimos el prompt a mano para cada informe. Definimos una **plantilla** con huecos (`{informe}`, `{destinatario}`) y la rellenamos con datos distintos cada vez. En Langflow esto será el componente *Prompt Template*.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

plantilla_resumen = ChatPromptTemplate.from_messages([
    ("system", "Eres un asistente clínico. Redactas resúmenes fieles al informe original, en español, "
               "adaptados al destinatario. No inventes datos que no aparezcan en el informe."),
    ("human", "Destinatario: {destinatario}\n\nInforme:\n{informe}\n\nRedacta un resumen de un máximo de tres frases."),
])

# La plantilla rellenada es una lista de mensajes, igual que en el bloque 1
prompt_rellenado = plantilla_resumen.invoke({"informe": INFORMES[1]["texto"], "destinatario": "el paciente"})
for m in prompt_rellenado.messages:
    print(f"[{m.type}] {m.content}\n")

[system] Eres un asistente clínico. Redactas resúmenes fieles al informe original, en español, adaptados al destinatario. No inventes datos que no aparezcan en el informe.

[human] Destinatario: el paciente

Informe:
Mujer de 45 años con antecedentes personales de asma bronquial. Consulta por tos y fiebre de tres días. La radiografía de tórax no muestra condensaciones, por lo que se descarta neumonía. Juicio clínico: bronquitis aguda.

Redacta un resumen de un máximo de tres frases.



Ahora reutilizamos la misma plantilla con varios informes y dos destinatarios distintos.

In [ ]:
destinatarios = ["el paciente, sin tecnicismos", "su médico de atención primaria"]

for informe in INFORMES[1:3]:
    print(f"===== Informe {informe['id']} =====")
    for dest in destinatarios:
        prompt = plantilla_resumen.invoke({"informe": informe["texto"], "destinatario": dest})
        print(f"--> Para {dest}:\n{llm.invoke(prompt).content}\n")

===== Informe 2 =====
--> Para el paciente, sin tecnicismos:
Usted, una mujer de 45 años con antecedentes de asma, consultó por tos y fiebre que han durado tres días. La radiografía de tórax no mostró signos de neumonía. El diagnóstico es bronquitis aguda, una inflamación de las vías respiratorias.

--> Para su médico de atención primaria:
Paciente mujer de 45 años con antecedentes de asma bronquial, consulta por tos y fiebre de tres días. La radiografía de tórax no revela condensaciones, descartando neumonía. Se diagnostica de bronquitis aguda.

===== Informe 3 =====
--> Para el paciente, sin tecnicismos:
Usted tiene un fibroadenoma en la mama izquierda que fue detectado hace dos años y no ha cambiado desde entonces. Su control actual muestra que sigue estable. Además, su madre y tía materna tuvieron cáncer de mama.

--> Para su médico de atención primaria:
Paciente mujer de 52 años con antecedentes de fibroadenoma en la mama izquierda diagnosticado por ecografía hace dos años, actual

✏️ **Prueba tú.**

1. Añade un tercer destinatario (por ejemplo, "un servicio de codificación clínica" o "un familiar del paciente").
2. Añade una nueva variable a la plantilla, por ejemplo `{idioma}`, y úsala para pedir el resumen en inglés o en gallego.
3. Revisa los resúmenes para el paciente del informe 3: ¿queda claro que el cáncer de mama lo tuvieron la madre y la tía, y no la paciente?

In [ ]:
destinatarios = ["el paciente, sin tecnicismos", "su médico de atención primaria", "un familiar del paciente"
            ]

for informe in INFORMES[1:3]:
    print(f"===== Informe {informe['id']} =====")
    for dest in destinatarios:
        prompt = plantilla_resumen.invoke({"informe": informe["texto"], "destinatario": dest})
        print(f"--> Para {dest}:\n{llm.invoke(prompt).content}\n")

===== Informe 2 =====
--> Para el paciente, sin tecnicismos:
Usted, una mujer de 45 años con antecedentes de asma, consultó por tos y fiebre que llevan tres días. La radiografía de tórax no mostró signos de neumonía. El diagnóstico es bronquitis aguda, una inflamación de las vías respiratorias.

--> Para su médico de atención primaria:
Paciente mujer de 45 años con antecedentes de asma bronquial, consulta por tos y fiebre de tres días. La radiografía de tórax no revela condensaciones, descartando neumonía. Se diagnostica de bronquitis aguda.

--> Para un familiar del paciente:
Se trata de una mujer de 45 años con antecedentes de asma bronquial que consultó por tos y fiebre durante tres días. La radiografía de tórax no mostró signos de neumonía, por lo que se descartó esa enfermedad. El diagnóstico clínico fue de bronquitis aguda.

===== Informe 3 =====
--> Para el paciente, sin tecnicismos:
Usted tiene un fibroadenoma en la mama izquierda que fue detectado hace dos años y no ha cambiad

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

plantilla_resumen = ChatPromptTemplate.from_messages([
    ("system", "Eres un asistente clínico. Redactas resúmenes fieles al informe original, "
               "adaptados al destinatario. No inventes datos que no aparezcan en el informe, escribe tu respuesta en el idioma:{idioma}."),
    ("human", "Destinatario: {destinatario}\n\nInforme:\n{informe}\n\nRedacta un resumen de un máximo de tres frases."),
])
# La plantilla rellenada es una lista de mensajes, igual que en el bloque 1
prompt_rellenado = plantilla_resumen.invoke({"informe": INFORMES[1]["texto"], "destinatario": "el paciente", "idioma":"gallego"})
for m in prompt_rellenado.messages:
    print(f"[{m.type}] {m.content}\n")
destinatarios = ["el paciente, sin tecnicismos", "su médico de atención primaria"]

for informe in INFORMES[1:3]:
    print(f"===== Informe {informe['id']} =====")
    for dest in destinatarios:
        prompt = plantilla_resumen.invoke({"informe": informe["texto"], "destinatario":dest,"idioma":"gallego"})
        print(f"--> Para {dest}:\n{llm.invoke(prompt).content}\n")

[system] Eres un asistente clínico. Redactas resúmenes fieles al informe original, adaptados al destinatario. No inventes datos que no aparezcan en el informe, escribe tu respuesta en el idioma:gallego.

[human] Destinatario: el paciente

Informe:
Mujer de 45 años con antecedentes personales de asma bronquial. Consulta por tos y fiebre de tres días. La radiografía de tórax no muestra condensaciones, por lo que se descarta neumonía. Juicio clínico: bronquitis aguda.

Redacta un resumen de un máximo de tres frases.

===== Informe 2 =====
--> Para el paciente, sin tecnicismos:
Ten unha muller de 45 anos con asma que leva tres días con tos e febre. As probas de raios X do peito non amosan sinais de neumonía. O seu diagnóstico é de bronquitis aguda.

--> Para su médico de atención primaria:
Muller de 45 anos con antecedentes de asma bronquial presenta unha clínica de tos e febre de tres días. A radiografía de tórax non revela condensacións, descartando neumonía. O diagnóstico clínico é de b

Si queda claro.

---
## Bloque 3 · Temperatura

En la presentación vimos que el modelo no elige una palabra: calcula una **distribución de probabilidad** sobre el siguiente token y luego muestrea de ella. La **temperatura** controla cuánto se aplana esa distribución:

- **Temperatura baja (≈ 0):** casi siempre se elige el token más probable. Respuestas estables y repetibles.
- **Temperatura alta (≈ 1–1.5):** se eligen con más frecuencia tokens menos probables. Respuestas más variadas, y también más propensas a errores.

Vamos a lanzar **cinco veces el mismo prompt** con dos temperaturas y comparar las respuestas. Usamos `.batch()`, que envía varias peticiones a la vez (lo veremos en detalle en el bloque 5).

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", None)

plantilla_titulo = ChatPromptTemplate.from_messages([
    ("human", "Escribe un título de menos de diez palabras para este informe clínico. "
              "Responde solo con el título.\n\n{informe}"),
])

N_REPETICIONES = 5
TEMPERATURAS = [0.0, 1.5]
entrada = {"informe": INFORMES[4]["texto"]}

resultados = {}
for t in TEMPERATURAS:
    cadena = plantilla_titulo | crear_modelo(temperatura=t)
    respuestas = cadena.batch([entrada] * N_REPETICIONES, config={"max_concurrency": 2})
    resultados[f"T = {t}"] = [r.content.strip() for r in respuestas]

tabla_temp = pd.DataFrame(resultados, index=[f"Ejecución {i+1}" for i in range(N_REPETICIONES)])
display(tabla_temp)
for col in tabla_temp:
    print(f"{col}: {tabla_temp[col].nunique()} respuestas distintas de {N_REPETICIONES}")

,T = 0.0,T = 1.5
Ejecución 1,Descompensación en paciente anciana con insuficiencia cardíaca y fibrilación,Caso de Insuficiencia Cardíaca Descompensada enPaciente VIH+ con Fibrilación
Ejecución 2,Descompensación en paciente anciana con insuficiencia cardíaca y fibrilación,Descripción de un caso clínico: insuficiencia cardiaca enuringada mexicana
Ejecución 3,Descompensación en paciente anciana con insuficiencia cardiaca y fibrilación,Paciente geriátrica con insuficiencia cardíaca y fibrilación auricular
Ejecución 4,Descompensación en paciente anciana con insuficiencia cardíaca y fibrilación,Caso clínico: insuficiencia cardíaca en paciente mayor con fibrilación uusia
Ejecución 5,Descompensación en paciente anciana con insuficiencia cardíaca y fibrilación,Paciente anciana con insuficiencia cardiaca y fibrilación auricular


T = 0.0: 2 respuestas distintas de 5
T = 1.5: 5 respuestas distintas de 5


✏️ **Prueba tú.**

1. Cambia el prompt para que la tarea sea **extraer** algo concreto (por ejemplo, "¿qué fármaco toma la paciente?"). ¿Se nota igual la diferencia entre temperaturas?
2. Sube la temperatura a `2.0` (máximo permitido). ¿Siguen siendo correctas las respuestas?
3. ¿Obtienes siempre la misma respuesta con `T = 0`? La temperatura 0 no garantiza un resultado idéntico en todas las ejecuciones; anota lo que observes para las preguntas finales.

**Para pensar:** ¿qué temperatura usarías para extraer diagnósticos de un informe? ¿Y para generar un texto educativo para pacientes?

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", None)

plantilla_titulo = ChatPromptTemplate.from_messages([
    ("human", "¿Qué fármaco toma la paciente según el siguiente informe clínico? "
              "Responde de forma breve y concisa.\n\n{informe}")
])

N_REPETICIONES = 5
TEMPERATURAS = [0.0, 1.5]
entrada = {"informe": INFORMES[4]["texto"]}

resultados = {}
for t in TEMPERATURAS:
    cadena = plantilla_titulo | crear_modelo(temperatura=t)
    respuestas = cadena.batch([entrada] * N_REPETICIONES, config={"max_concurrency": 2})
    resultados[f"T = {t}"] = [r.content.strip() for r in respuestas]

tabla_temp = pd.DataFrame(resultados, index=[f"Ejecución {i+1}" for i in range(N_REPETICIONES)])
display(tabla_temp)
for col in tabla_temp:
    print(f"{col}: {tabla_temp[col].nunique()} respuestas distintas de {N_REPETICIONES}")

,T = 0.0,T = 1.5
Ejecución 1,"No se especifica en el informe, pero probablemente toma un anticoagulante (como warfarina o NOAC) para la fibrilación auricular.",No se especifica el fármaco que toma la paciente en el informe clínico.
Ejecución 2,"No se especifica en el informe, pero probablemente toma un anticoagulante (como warfarina o NOAC) para la fibrilación auricular.","El informe no especifica qué fármaco en particular toma la paciente, solo indica que está en tratamiento anticoagulante para la fibrilación auricular."
Ejecución 3,"No se especifica en el informe, pero probablemente toma un anticoagulante (como warfarina o NOAC) para la fibrilación auricular.","La paciente probablemente está tomando warfarina o un anticoagulante oral directo, como apixaban o rivaroxabán, para fibrilación auricular."
Ejecución 4,"No se especifica en el informe, pero probablemente toma un anticoagulante (como warfarina o NOAC) para la fibrilación auricular.",El informe no proporciona información específica sobre el fármaco que está tomando la paciente.
Ejecución 5,"No se especifica en el informe, pero probablemente toma un anticoagulante (como warfarina o NOAC) para la fibrilación auricular.","No se especifica en el informe, pero es probable que esté tomando un antiarrhythmic, un vasodilatador o un diurético, además del anticoagulante, pero eso no se puede afirmar con certeza a partir del texto proporcionado."


T = 0.0: 1 respuestas distintas de 5
T = 1.5: 5 respuestas distintas de 5


In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", None)

plantilla_titulo = ChatPromptTemplate.from_messages([
    ("human", "¿Qué fármaco toma la paciente según el siguiente informe clínico? "
              "Responde de forma breve y concisa.\n\n{informe}")
])

N_REPETICIONES = 5
TEMPERATURAS = [0.0, 1.5,2.0]
entrada = {"informe": INFORMES[4]["texto"]}

resultados = {}
for t in TEMPERATURAS:
    cadena = plantilla_titulo | crear_modelo(temperatura=t)
    respuestas = cadena.batch([entrada] * N_REPETICIONES, config={"max_concurrency": 2})
    resultados[f"T = {t}"] = [r.content.strip() for r in respuestas]

tabla_temp = pd.DataFrame(resultados, index=[f"Ejecución {i+1}" for i in range(N_REPETICIONES)])
display(tabla_temp)
for col in tabla_temp:
    print(f"{col}: {tabla_temp[col].nunique()} respuestas distintas de {N_REPETICIONES}")

T = 0.0  \
Ejecución 1  No se especifica en el informe, pero probablemente toma un anticoagulante (como warfarina o NOAC) para la fibrilación auricular.   
Ejecución 2  No se especifica en el informe, pero probablemente toma un anticoagulante (como warfarina o NOAC) para su fibrilación auricular.   
Ejecución 3  No se especifica en el informe, pero probablemente toma un anticoagulante (como warfarina o NOAC) para la fibrilación auricular.   
Ejecución 4  No se especifica en el informe, pero probablemente toma un anticoagulante (como warfarina o NOAC) para la fibrilación auricular.   
Ejecución 5  No se especifica en el informe, pero probablemente toma un anticoagulante (como warfarina o NOAC) para su fibrilación auricular.   

                                                                                                                                                          T = 1.5  \
Ejecución 1                                                                                               La paciente está tomando acenocumarol (anticoagulante).   
Ejecución 2  El informe no especifica un farmaco concreto que tome la paciente, solo indica que está en tratamiento anticoagulante para su fibrilación auricular.   
Ejecución 3                                                                              El fármaco que probablemente toma la paciente es un anticoagulante oral.   
Ejecución 4                                                               El archivo no proporciona información específica sobre el fármaco que toma la paciente.   
Ejecución 5                                                                                                               Tratamiento con fármaco anticoagulante.   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                

T = 0.0: 2 respuestas distintas de 5
T = 1.5: 5 respuestas distintas de 5
T = 2.0: 5 respuestas distintas de 5


NO, en temperaturas muy altas da respuestas sin sentido.


---
## Bloque 4 · Salida estructurada

Hasta ahora el modelo devolvía texto libre, que es cómodo de leer pero difícil de procesar: no se puede guardar en una base de datos, ni contar, ni comparar con una anotación. Para integrar un LLM en un sistema de información clínica necesitamos que devuelva **datos con un formato fijo**.

Para ello definimos un **esquema** con [Pydantic](https://docs.pydantic.dev): una clase de Python que describe qué campos queremos y de qué tipo son. Las descripciones de los campos (`description=...`) también forman parte del prompt: el modelo las lee para saber qué poner en cada campo.

In [ ]:
from pydantic import BaseModel, Field

class Diagnostico(BaseModel):
    diagnostico: str = Field(description="Nombre de la enfermedad o diagnóstico, tal como aparece en el texto")
    negado: bool = Field(description="True si el texto indica que el diagnóstico se descarta o no está presente")
    antecedente_familiar: bool = Field(description="True si el diagnóstico corresponde a un familiar y no al paciente")

class ExtraccionDiagnosticos(BaseModel):
    diagnosticos: list[Diagnostico] = Field(description="Todos los diagnósticos mencionados en el informe")

Con `with_structured_output` le pedimos al modelo que su respuesta cumpla el esquema. LangChain se encarga de enviar el esquema al proveedor y de convertir la respuesta en un objeto Python.

In [ ]:
INSTRUCCIONES_EXTRACCION = (
    "Eres un experto en documentación clínica. Extrae del informe todas las enfermedades y diagnósticos "
    "mencionados, tanto del paciente como de sus familiares, incluidos los que se descartan. "
    "No incluyas síntomas aislados, signos de la exploración ni resultados de pruebas."
)

plantilla_extraccion = ChatPromptTemplate.from_messages([
    ("system", INSTRUCCIONES_EXTRACCION),
    ("human", "Informe:\n{informe}"),
])

llm_estructurado = llm.with_structured_output(ExtraccionDiagnosticos)

informe = INFORMES[0]
resultado = llm_estructurado.invoke(plantilla_extraccion.invoke({"informe": informe["texto"]}))

print("Tipo de objeto:", type(resultado).__name__, "\n")
print(informe["texto"], "\n")
display(pd.DataFrame([d.model_dump() for d in resultado.diagnosticos]))
# model_dump() es un método de Pydantic que convierte un objeto Pydantic en un diccionario de Python

Tipo de objeto: ExtraccionDiagnosticos 

Varón de 67 años con hipertensión arterial y diabetes mellitus tipo 2 que acude por dolor torácico. ECG sin alteraciones y troponinas normales; se descarta infarto agudo de miocardio. Padre fallecido por cáncer de colon. 



,diagnostico,negado,antecedente_familiar
0,hipertensión arterial,False,False
1,diabetes mellitus tipo 2,False,False
2,infarto agudo de miocardio,True,False
3,cáncer de colon,False,True


El resultado ya no es texto: es un objeto con atributos que podemos recorrer, filtrar o guardar. Por ejemplo, la lista de problemas **activos del paciente** son los diagnósticos no negados y no familiares:

In [ ]:
activos = [d.diagnostico for d in resultado.diagnosticos if not d.negado and not d.antecedente_familiar]
print("Problemas activos del paciente:", activos)

Problemas activos del paciente: ['hipertensión arterial', 'diabetes mellitus tipo 2']


In [ ]:
# Y el objeto se puede convertir también directamente en JSON, listo para otro sistema
print(resultado.model_dump_json(indent=2))

{
  "diagnosticos": [
    {
      "diagnostico": "hipertensión arterial",
      "negado": false,
      "antecedente_familiar": false
    },
    {
      "diagnostico": "diabetes mellitus tipo 2",
      "negado": false,
      "antecedente_familiar": false
    },
    {
      "diagnostico": "infarto agudo de miocardio",
      "negado": true,
      "antecedente_familiar": false
    },
    {
      "diagnostico": "cáncer de colon",
      "negado": false,
      "antecedente_familiar": true
    }
  ]
}


Probemos con los informes más difíciles. El informe 7 contiene una **sospecha** (no confirmada) y un **antecedente familiar negado**; el informe 10 menciona la misma enfermedad en la paciente y en su madre.

In [ ]:
for informe in [INFORMES[6], INFORMES[9]]:
    print(f"===== Informe {informe['id']} =====\n{informe['texto']}\n")
    resultado = llm_estructurado.invoke(plantilla_extraccion.invoke({"informe": informe["texto"]}))
    display(pd.DataFrame([d.model_dump() for d in resultado.diagnosticos]))

===== Informe 7 =====
Varón de 59 años con EPOC y obesidad. Se solicita polisomnografía por sospecha de síndrome de apnea obstructiva del sueño. Sin antecedentes familiares de enfermedad cardiovascular.



,diagnostico,negado,antecedente_familiar
0,EPOC,False,False
1,obesidad,False,False
2,síndrome de apnea obstructiva del sueño,True,False


===== Informe 10 =====
Mujer de 25 años que consulta por cefalea intensa. Exploración neurológica normal. Se descarta hemorragia subaracnoidea mediante TC craneal. Juicio clínico: migraña sin aura. Antecedentes familiares de migraña por línea materna.



,diagnostico,negado,antecedente_familiar
0,hemorragia subaracnoidea,True,False
1,migraña sin aura,False,False
2,migraña por línea materna,False,True


✏️ **Prueba tú.**

1. **Añade un campo al esquema** `Diagnostico`, por ejemplo `sospecha: bool` ("True si el diagnóstico no está confirmado") o `familiar: str` ("parentesco del familiar, o cadena vacía"). Vuelve a ejecutar las celdas del bloque y comprueba cómo lo rellena con el informe 7.
2. Cambia la **descripción** del campo `negado` por algo vago, como `"negado"`. ¿Empeora la extracción?
3. Elimina de `INSTRUCCIONES_EXTRACCION` la frase sobre síntomas. ¿Aparece "dolor torácico" o "cefalea" como diagnóstico?

> Si modificas el esquema, recuerda volver a ejecutar la celda de `llm_estructurado = ...` para que el modelo use la nueva versión.

In [ ]:
from pydantic import BaseModel, Field

class Diagnostico(BaseModel):
    diagnostico: str = Field(description="Nombre de la enfermedad o diagnóstico, tal como aparece en el texto")
    negado: bool = Field(description="True si el texto indica que el diagnóstico se descarta o no está presente")
    antecedente_familiar: bool = Field(description="True si el diagnóstico corresponde a un familiar y no al paciente")
    sospecha: bool =Field(description=" devuelve True si el diagnostico no esta confirmado")

class ExtraccionDiagnosticos(BaseModel):
    diagnosticos: list[Diagnostico] = Field(description="Todos los diagnósticos mencionados en el informe")

INSTRUCCIONES_EXTRACCION = (
    "Eres un experto en documentación clínica. Extrae del informe todas las enfermedades y diagnósticos "
    "mencionados, tanto del paciente como de sus familiares, incluidos los que se descartan. "
    "No incluyas síntomas aislados, signos de la exploración ni resultados de pruebas."
)

plantilla_extraccion = ChatPromptTemplate.from_messages([
    ("system", INSTRUCCIONES_EXTRACCION),
    ("human", "Informe:\n{informe}"),
])

llm_estructurado = llm.with_structured_output(ExtraccionDiagnosticos)

informe = INFORMES[0]
resultado = llm_estructurado.invoke(plantilla_extraccion.invoke({"informe": informe["texto"]}))

print("Tipo de objeto:", type(resultado).__name__, "\n")
print(informe["texto"], "\n")
display(pd.DataFrame([d.model_dump() for d in resultado.diagnosticos]))
# model_dump() es un método de Pydantic que convierte un objeto Pydantic en un diccionario de Python

activos = [d.diagnostico for d in resultado.diagnosticos if not d.negado and not d.antecedente_familiar]
print("Problemas activos del paciente:", activos)

# Y el objeto se puede convertir también directamente en JSON, listo para otro sistema
print(resultado.model_dump_json(indent=2))

for informe in [INFORMES[6], INFORMES[9]]:
    print(f"===== Informe {informe['id']} =====\n{informe['texto']}\n")
    resultado = llm_estructurado.invoke(plantilla_extraccion.invoke({"informe": informe["texto"]}))
    display(pd.DataFrame([d.model_dump() for d in resultado.diagnosticos]))

Tipo de objeto: ExtraccionDiagnosticos 

Varón de 67 años con hipertensión arterial y diabetes mellitus tipo 2 que acude por dolor torácico. ECG sin alteraciones y troponinas normales; se descarta infarto agudo de miocardio. Padre fallecido por cáncer de colon. 



,diagnostico,negado,antecedente_familiar,sospecha
0,hipertensión arterial,False,False,False
1,diabetes mellitus tipo 2,False,False,False
2,infarto agudo de miocardio,True,False,False
3,cáncer de colon,False,True,False


Problemas activos del paciente: ['hipertensión arterial', 'diabetes mellitus tipo 2']
{
  "diagnosticos": [
    {
      "diagnostico": "hipertensión arterial",
      "negado": false,
      "antecedente_familiar": false,
      "sospecha": false
    },
    {
      "diagnostico": "diabetes mellitus tipo 2",
      "negado": false,
      "antecedente_familiar": false,
      "sospecha": false
    },
    {
      "diagnostico": "infarto agudo de miocardio",
      "negado": true,
      "antecedente_familiar": false,
      "sospecha": false
    },
    {
      "diagnostico": "cáncer de colon",
      "negado": false,
      "antecedente_familiar": true,
      "sospecha": false
    }
  ]
}
===== Informe 7 =====
Varón de 59 años con EPOC y obesidad. Se solicita polisomnografía por sospecha de síndrome de apnea obstructiva del sueño. Sin antecedentes familiares de enfermedad cardiovascular.



,diagnostico,negado,antecedente_familiar,sospecha
0,EPOC,False,False,False
1,obesidad,False,False,False
2,síndrome de apnea obstructiva del sueño,True,False,True


===== Informe 10 =====
Mujer de 25 años que consulta por cefalea intensa. Exploración neurológica normal. Se descarta hemorragia subaracnoidea mediante TC craneal. Juicio clínico: migraña sin aura. Antecedentes familiares de migraña por línea materna.



,diagnostico,negado,antecedente_familiar,sospecha
0,hemorragia subaracnoidea,True,False,False
1,migraña sin aura,False,False,False
2,migraña,False,True,False


In [ ]:
from pydantic import BaseModel, Field

class Diagnostico(BaseModel):
    diagnostico: str = Field(description="Nombre de la enfermedad o diagnóstico, tal como aparece en el texto")
    negado: bool = Field(description="negado")
    antecedente_familiar: bool = Field(description="True si el diagnóstico corresponde a un familiar y no al paciente")
    sospecha: bool =Field(description=" devuelve True si el diagnostico no esta confirmado")

class ExtraccionDiagnosticos(BaseModel):
    diagnosticos: list[Diagnostico] = Field(description="Todos los diagnósticos mencionados en el informe")

INSTRUCCIONES_EXTRACCION = (
    "Eres un experto en documentación clínica. Extrae del informe todas las enfermedades y diagnósticos "
    "mencionados, tanto del paciente como de sus familiares, incluidos los que se descartan. "
    "No incluyas síntomas aislados, signos de la exploración ni resultados de pruebas."
)

plantilla_extraccion = ChatPromptTemplate.from_messages([
    ("system", INSTRUCCIONES_EXTRACCION),
    ("human", "Informe:\n{informe}"),
])

llm_estructurado = llm.with_structured_output(ExtraccionDiagnosticos)

informe = INFORMES[0]
resultado = llm_estructurado.invoke(plantilla_extraccion.invoke({"informe": informe["texto"]}))

print("Tipo de objeto:", type(resultado).__name__, "\n")
print(informe["texto"], "\n")
display(pd.DataFrame([d.model_dump() for d in resultado.diagnosticos]))
# model_dump() es un método de Pydantic que convierte un objeto Pydantic en un diccionario de Python

activos = [d.diagnostico for d in resultado.diagnosticos if not d.negado and not d.antecedente_familiar]
print("Problemas activos del paciente:", activos)

# Y el objeto se puede convertir también directamente en JSON, listo para otro sistema
print(resultado.model_dump_json(indent=2))

for informe in [INFORMES[6], INFORMES[9]]:
    print(f"===== Informe {informe['id']} =====\n{informe['texto']}\n")
    resultado = llm_estructurado.invoke(plantilla_extraccion.invoke({"informe": informe["texto"]}))
    display(pd.DataFrame([d.model_dump() for d in resultado.diagnosticos]))

Tipo de objeto: ExtraccionDiagnosticos 

Varón de 67 años con hipertensión arterial y diabetes mellitus tipo 2 que acude por dolor torácico. ECG sin alteraciones y troponinas normales; se descarta infarto agudo de miocardio. Padre fallecido por cáncer de colon. 



,diagnostico,negado,antecedente_familiar,sospecha
0,hipertensión arterial,False,False,False
1,diabetes mellitus tipo 2,False,False,False
2,infarto agudo de miocardio,True,False,False
3,cáncer de colon,False,True,False


Problemas activos del paciente: ['hipertensión arterial', 'diabetes mellitus tipo 2']
{
  "diagnosticos": [
    {
      "diagnostico": "hipertensión arterial",
      "negado": false,
      "antecedente_familiar": false,
      "sospecha": false
    },
    {
      "diagnostico": "diabetes mellitus tipo 2",
      "negado": false,
      "antecedente_familiar": false,
      "sospecha": false
    },
    {
      "diagnostico": "infarto agudo de miocardio",
      "negado": true,
      "antecedente_familiar": false,
      "sospecha": false
    },
    {
      "diagnostico": "cáncer de colon",
      "negado": false,
      "antecedente_familiar": true,
      "sospecha": false
    }
  ]
}
===== Informe 7 =====
Varón de 59 años con EPOC y obesidad. Se solicita polisomnografía por sospecha de síndrome de apnea obstructiva del sueño. Sin antecedentes familiares de enfermedad cardiovascular.



,diagnostico,negado,antecedente_familiar,sospecha
0,EPOC,False,False,False
1,obesidad,False,False,False
2,síndrome de apnea obstructiva del sueño,False,False,True


===== Informe 10 =====
Mujer de 25 años que consulta por cefalea intensa. Exploración neurológica normal. Se descarta hemorragia subaracnoidea mediante TC craneal. Juicio clínico: migraña sin aura. Antecedentes familiares de migraña por línea materna.



,diagnostico,negado,antecedente_familiar,sospecha
0,hemorragia subaracnoidea,True,False,False
1,migraña sin aura,False,False,False
2,migraña por línea materna,False,True,False


In [ ]:
 Al pasar de zero-shot a few-shot el modelo no mejoró, sino que empeoró ligeramente: de 26 a 24-25 diagnósticos correctos de 31. Aparecieron nuevos errores en el informe 1 (cáncer de colon, atributo erróneo) y, con los dos ejemplos, en el informe 4 (apendicitis aguda, omitido). Los errores se mantuvieron en los informes 3, 6, 7 y 10. Con solo 10 informes, una diferencia de 1-2 aciertos es muy pequeña y puede deberse al azar, y los ejemplos añadidos no cubrían bien los casos difícilesfrom pydantic import BaseModel, Field

class Diagnostico(BaseModel):
    diagnostico: str = Field(description="Nombre de la enfermedad o diagnóstico, tal como aparece en el texto")
    negado: bool = Field(description="True si el texto indica que el diagnóstico se descarta o no está presente")
    antecedente_familiar: bool = Field(description="True si el diagnóstico corresponde a un familiar y no al paciente")
    sospecha: bool =Field(description=" devuelve True si el diagnostico no esta confirmado")

class ExtraccionDiagnosticos(BaseModel):
    diagnosticos: list[Diagnostico] = Field(description="Todos los diagnósticos mencionados en el informe")

INSTRUCCIONES_EXTRACCION = (
    "Eres un experto en documentación clínica. Extrae del informe todas las enfermedades y diagnósticos "
    "mencionados, tanto del paciente como de sus familiares, incluidos los que se descartan. "
)

plantilla_extraccion = ChatPromptTemplate.from_messages([
    ("system", INSTRUCCIONES_EXTRACCION),
    ("human", "Informe:\n{informe}"),
])

llm_estructurado = llm.with_structured_output(ExtraccionDiagnosticos)

informe = INFORMES[0]
resultado = llm_estructurado.invoke(plantilla_extraccion.invoke({"informe": informe["texto"]}))

print("Tipo de objeto:", type(resultado).__name__, "\n")
print(informe["texto"], "\n")
display(pd.DataFrame([d.model_dump() for d in resultado.diagnosticos]))
# model_dump() es un método de Pydantic que convierte un objeto Pydantic en un diccionario de Python

activos = [d.diagnostico for d in resultado.diagnosticos if not d.negado and not d.antecedente_familiar]
print("Problemas activos del paciente:", activos)

# Y el objeto se puede convertir también directamente en JSON, listo para otro sistema
print(resultado.model_dump_json(indent=2))

for informe in [INFORMES[6], INFORMES[9]]:
    print(f"===== Informe {informe['id']} =====\n{informe['texto']}\n")
    resultado = llm_estructurado.invoke(plantilla_extraccion.invoke({"informe": informe["texto"]}))
    display(pd.DataFrame([d.model_dump() for d in resultado.diagnosticos]))

Tipo de objeto: ExtraccionDiagnosticos 

Varón de 67 años con hipertensión arterial y diabetes mellitus tipo 2 que acude por dolor torácico. ECG sin alteraciones y troponinas normales; se descarta infarto agudo de miocardio. Padre fallecido por cáncer de colon. 



,diagnostico,negado,antecedente_familiar,sospecha
0,hipertensión arterial,False,False,False
1,diabetes mellitus tipo 2,False,False,False
2,infarto agudo de miocardio,True,False,False
3,cáncer de colon,False,True,False


Problemas activos del paciente: ['hipertensión arterial', 'diabetes mellitus tipo 2']
{
  "diagnosticos": [
    {
      "diagnostico": "hipertensión arterial",
      "negado": false,
      "antecedente_familiar": false,
      "sospecha": false
    },
    {
      "diagnostico": "diabetes mellitus tipo 2",
      "negado": false,
      "antecedente_familiar": false,
      "sospecha": false
    },
    {
      "diagnostico": "infarto agudo de miocardio",
      "negado": true,
      "antecedente_familiar": false,
      "sospecha": false
    },
    {
      "diagnostico": "cáncer de colon",
      "negado": false,
      "antecedente_familiar": true,
      "sospecha": false
    }
  ]
}
===== Informe 7 =====
Varón de 59 años con EPOC y obesidad. Se solicita polisomnografía por sospecha de síndrome de apnea obstructiva del sueño. Sin antecedentes familiares de enfermedad cardiovascular.



,diagnostico,negado,antecedente_familiar,sospecha
0,EPOC,False,False,False
1,obesidad,False,False,False
2,síndrome de apnea obstructiva del sueño,False,False,True


===== Informe 10 =====
Mujer de 25 años que consulta por cefalea intensa. Exploración neurológica normal. Se descarta hemorragia subaracnoidea mediante TC craneal. Juicio clínico: migraña sin aura. Antecedentes familiares de migraña por línea materna.



,diagnostico,negado,antecedente_familiar,sospecha
0,hemorragia subaracnoidea,True,False,False
1,migraña sin aura,False,False,False
2,migraña,False,True,False


---
## Bloque 5 · Cadenas, procesamiento por lotes y evaluación

### 5.1 Una cadena `prompt | llm | parser`

LangChain permite unir piezas con el operador `|`: la salida de cada una es la entrada de la siguiente. Una cadena típica tiene tres eslabones:

1. **prompt**: rellena la plantilla con los datos.
2. **llm**: llama al modelo.
3. **parser**: convierte el texto de la respuesta en un objeto Python.

En el bloque 4, `with_structured_output` hacía los pasos 2 y 3 a la vez usando una función nativa del proveedor. Aquí usamos un `PydanticOutputParser` explícito, que añade al prompt unas instrucciones de formato y luego valida la respuesta contra el mismo esquema. Así se ven los tres eslabones por separado, igual que los verás después como tres cajas en Langflow.

In [ ]:
from langchain_core.output_parsers import PydanticOutputParser

parser = PydanticOutputParser(pydantic_object=ExtraccionDiagnosticos)

plantilla_cadena = ChatPromptTemplate.from_messages([
    ("system", INSTRUCCIONES_EXTRACCION + "\n\n{instrucciones_formato}"),
    ("human", "Informe:\n{informe}"),
]).partial(instrucciones_formato=parser.get_format_instructions())

cadena = plantilla_cadena | llm | parser
# Una sola llamada

print(cadena.invoke({"informe": INFORMES[1]["texto"]}))

diagnosticos=[Diagnostico(diagnostico='asma bronquial', negado=False, antecedente_familiar=False, sospecha=False), Diagnostico(diagnostico='neumonía', negado=True, antecedente_familiar=False, sospecha=False), Diagnostico(diagnostico='bronquitis aguda', negado=False, antecedente_familiar=False, sospecha=False)]


In [ ]:
print(parser.get_format_instructions())

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"$defs": {"Diagnostico": {"properties": {"diagnostico": {"description": "Nombre de la enfermedad o diagnóstico, tal como aparece en el texto", "title": "Diagnostico", "type": "string"}, "negado": {"description": "True si el texto indica que el diagnóstico se descarta o no está presente", "title": "Negado", "type": "boolean"}, "antecedente_familiar": {"description": "True si el diagnóstico corresponde a un familiar y no al paciente", "title": "Antecedente Familiar", "type": "boolean"}, "sospecha": {"description": " devuelve True si el diagnost

### 5.2 Procesamiento por lotes con `.batch()`

`.batch()` ejecuta la cadena sobre una lista de entradas, con varias peticiones en paralelo. Con `return_exceptions=True` un error en un informe (por ejemplo, una respuesta que no cumple el esquema) no detiene el resto.

> Si usas el plan gratuito de Google y aparecen errores de límite de peticiones (*429, rate limit*), baja `max_concurrency` a 1.

In [ ]:
import time

entradas = [{"informe": inf["texto"]} for inf in INFORMES]

inicio = time.time()
predicciones = cadena.batch(entradas, config={"max_concurrency": 4}, return_exceptions=True)
print(f"{len(predicciones)} informes procesados en {time.time() - inicio:.1f} s")

errores = [inf["id"] for inf, p in zip(INFORMES, predicciones) if isinstance(p, Exception)]
print("Informes con error:", errores or "ninguno")

10 informes procesados en 2.4 s
Informes con error: ninguno


✏️ **Prueba tú.**

1. **Muestra el resultado** de uno de los informes. Recuerda que sigue la estructura Pydantic definida anteriormente.

In [ ]:
from langchain_core.output_parsers import PydanticOutputParser

parser = PydanticOutputParser(pydantic_object=ExtraccionDiagnosticos)

plantilla_cadena = ChatPromptTemplate.from_messages([
    ("system", INSTRUCCIONES_EXTRACCION + "\n\n{instrucciones_formato}"),
    ("human", "Informe:\n{informe}"),
]).partial(instrucciones_formato=parser.get_format_instructions())

cadena = plantilla_cadena | llm | parser

# Una sola llamada
print(cadena.invoke({"informe": INFORMES[2]["texto"]}))

diagnosticos=[Diagnostico(diagnostico='fibroadenoma de mama izquierda', negado=False, antecedente_familiar=False, sospecha=False), Diagnostico(diagnostico='cáncer de mama', negado=True, antecedente_familiar=True, sospecha=False)]


### 5.3 ¿Cuántos diagnósticos acierta?

Comparamos cada predicción con la anotación de referencia. Para cada diagnóstico anotado buscamos en la predicción uno con el mismo nombre (admitimos pequeñas diferencias de redacción y los sinónimos definidos al principio) y comprobamos sus dos atributos.

Clasificamos cada diagnóstico en:

- ✅ **correcto**: encontrado y con `negado` y `antecedente_familiar` bien.
- ⚠️ **atributo erróneo**: encontrado, pero con algún atributo mal. Es el error más peligroso: el sistema diría, por ejemplo, que el paciente tiene una neumonía que se ha descartado.
- ❌ **omitido**: el modelo no lo ha extraído.
- ➕ **sobrante**: el modelo extrae algo que no está en la referencia (un síntoma, una prueba o un diagnóstico inventado).

No hace falta que entiendas el código de evaluación en detalle: lo importante es la tabla de resultados.

In [ ]:
import unicodedata
from difflib import SequenceMatcher

def normalizar(texto):
    texto = unicodedata.normalize("NFKD", texto.lower()).encode("ascii", "ignore").decode()
    return " ".join(texto.replace(",", " ").replace(".", " ").split())

def similitud(pred, gold):
    # Puntuación entre 0 y 1 del parecido entre un nombre predicho y uno de referencia (o sus sinónimos)
    p = normalizar(pred)
    mejor = 0.0
    for nombre in [gold] + SINONIMOS.get(gold, []):
        g = normalizar(nombre)
        if p == g:
            return 1.0
        if g in p or p in g:
            mejor = max(mejor, 0.9)
        mejor = max(mejor, SequenceMatcher(None, p, g).ratio())
    return mejor

def evaluar_informe(informe, prediccion, umbral=0.8):
    gold = informe["gold"]
    pred = [] if isinstance(prediccion, Exception) else [d.model_dump() for d in prediccion.diagnosticos]

    # Emparejamiento voraz: primero las parejas más parecidas; a igual nombre, las que coinciden en atributos
    parejas = []
    for i, g in enumerate(gold):
        for j, p in enumerate(pred):
            s = similitud(p["diagnostico"], g["diagnostico"])
            if s >= umbral:
                coinciden = (p["negado"] == g["negado"]) + (p["antecedente_familiar"] == g["antecedente_familiar"])
                parejas.append((s + 0.01 * coinciden, i, j))
    parejas.sort(reverse=True)
    gold_usado, pred_usado, filas = {}, set(), []
    for _, i, j in parejas:
        if i not in gold_usado and j not in pred_usado:
            gold_usado[i] = j
            pred_usado.add(j)

    for i, g in enumerate(gold):
        fila = {"informe": informe["id"], "referencia": g["diagnostico"],
                "neg_ref": g["negado"], "fam_ref": g["antecedente_familiar"]}
        if i in gold_usado:
            p = pred[gold_usado[i]]
            ok = p["negado"] == g["negado"] and p["antecedente_familiar"] == g["antecedente_familiar"]
            fila.update(prediccion=p["diagnostico"], neg_pred=p["negado"], fam_pred=p["antecedente_familiar"],
                        resultado="✅ correcto" if ok else "⚠️ atributo erróneo")
        else:
            fila.update(prediccion="", neg_pred=None, fam_pred=None, resultado="❌ omitido")
        filas.append(fila)
    for j, p in enumerate(pred):
        if j not in pred_usado:
            filas.append({"informe": informe["id"], "referencia": "", "neg_ref": None, "fam_ref": None,
                          "prediccion": p["diagnostico"], "neg_pred": p["negado"], "fam_pred": p["antecedente_familiar"],
                          "resultado": "➕ sobrante"})
    return filas

def evaluar(predicciones, mostrar=True):
    tabla = pd.DataFrame([f for inf, p in zip(INFORMES, predicciones) for f in evaluar_informe(inf, p)])
    conteo = tabla["resultado"].value_counts()
    n_gold = sum(len(inf["gold"]) for inf in INFORMES)
    correctos = conteo.get("✅ correcto", 0)
    if mostrar:
        print(f"Diagnósticos correctos: {correctos} de {n_gold} ({100 * correctos / n_gold:.0f} %)")
        for etiqueta in ["⚠️ atributo erróneo", "❌ omitido", "➕ sobrante"]:
            print(f"  {etiqueta}: {conteo.get(etiqueta, 0)}")
    return tabla

In [ ]:
tabla_zero_shot = evaluar(predicciones)

Diagnósticos correctos: 26 de 31 (84 %)
  ⚠️ atributo erróneo: 2
  ❌ omitido: 3
  ➕ sobrante: 0


Veamos solo los casos en los que el modelo se ha equivocado:

In [ ]:
display(tabla_zero_shot[tabla_zero_shot["resultado"] != "✅ correcto"])

,informe,referencia,neg_ref,fam_ref,prediccion,neg_pred,fam_pred,resultado
8,3,cáncer de mama,False,True,cáncer de mama,True,True,⚠️ atributo erróneo
16,6,infección cutánea,True,False,,None,None,❌ omitido
20,7,enfermedad cardiovascular,True,True,,None,None,❌ omitido
29,10,migraña sin aura,False,False,migraña sin aura,False,True,⚠️ atributo erróneo
30,10,migraña,False,True,,None,None,❌ omitido


### 5.4 ¿Mejora el modelo si le damos ejemplos? (*few-shot*)

En la presentación vimos el **aprendizaje en contexto**: el modelo puede adaptar su comportamiento solo con ejemplos incluidos en el prompt, sin reentrenarlo. Hasta ahora hemos trabajado en modo *zero-shot* (solo instrucciones). Vamos a añadir dos ejemplos resueltos que muestren los casos difíciles y a comparar los resultados con la misma evaluación.

Los ejemplos **no** pertenecen a los 10 informes de evaluación: si lo fueran, estaríamos dándole las respuestas al modelo.

In [ ]:
EJEMPLOS = """
Ejemplos resueltos:

Informe: "Paciente de 60 años con artrosis de rodilla. Se descarta gota tras analizar el líquido sinovial. Hermano con artritis reumatoide."
Diagnósticos:
- artrosis de rodilla (negado: false, antecedente_familiar: false)
- gota (negado: true, antecedente_familiar: false)
- artritis reumatoide (negado: false, antecedente_familiar: true)

Informe: "Mujer de 40 años con hipotiroidismo. No hay antecedentes familiares de cáncer de tiroides. Refiere cansancio."
Diagnósticos:
- hipotiroidismo (negado: false, antecedente_familiar: false)
- cáncer de tiroides (negado: true, antecedente_familiar: true)
(El cansancio es un síntoma y no se incluye.)
"""

plantilla_few_shot = ChatPromptTemplate.from_messages([
    ("system", INSTRUCCIONES_EXTRACCION + "\n" + EJEMPLOS + "\n{instrucciones_formato}"),
    ("human", "Informe:\n{informe}"),
]).partial(instrucciones_formato=parser.get_format_instructions())

cadena_few_shot = plantilla_few_shot | llm | parser
predicciones_few_shot = cadena_few_shot.batch(entradas, config={"max_concurrency": 4}, return_exceptions=True)

tabla_few_shot = evaluar(predicciones_few_shot)

Diagnósticos correctos: 24 de 31 (77 %)
  ⚠️ atributo erróneo: 3
  ❌ omitido: 4
  ➕ sobrante: 0


In [ ]:
def resumen(tabla):
    return tabla["resultado"].value_counts()

comparacion = pd.DataFrame({"zero-shot": resumen(tabla_zero_shot), "few-shot": resumen(tabla_few_shot)}).fillna(0).astype(int)
display(comparacion)

display(tabla_few_shot[tabla_few_shot["resultado"] != "✅ correcto"])

,zero-shot,few-shot
resultado,,
✅ correcto,26,24
❌ omitido,3,4
⚠️ atributo erróneo,2,3


,informe,referencia,neg_ref,fam_ref,prediccion,neg_pred,fam_pred,resultado
3,1,cáncer de colon,False,True,cáncer de colon,True,True,⚠️ atributo erróneo
8,3,cáncer de mama,False,True,cáncer de mama,True,True,⚠️ atributo erróneo
9,4,apendicitis aguda,True,False,,None,None,❌ omitido
16,6,infección cutánea,True,False,,None,None,❌ omitido
20,7,enfermedad cardiovascular,True,True,,None,None,❌ omitido
29,10,migraña sin aura,False,False,migraña sin aura,False,True,⚠️ atributo erróneo
30,10,migraña,False,True,,None,None,❌ omitido


✏️ **Prueba tú.**

1. Ejecuta de nuevo la evaluación *zero-shot* y *few-shot*. ¿Son los resultados idénticos aunque la temperatura sea 0? ¿Qué implica esto para comparar dos prompts con solo 10 informes?
2. Cambia los ejemplos de `EJEMPLOS`: quita el que tiene un antecedente familiar negado y comprueba qué pasa con el informe 7.
3. Si has añadido el campo `sospecha` en el bloque 4, ¿cómo lo trata la evaluación? (Pista: la referencia no lo tiene anotado.)
4. Si tienes acceso a los dos proveedores, cambia `PROVEEDOR`, vuelve a ejecutar el notebook y compara los modelos con la misma evaluación.

In [ ]:
EJEMPLOS = """
Ejemplos resueltos:

Informe: "Paciente de 60 años con artrosis de rodilla. Se descarta gota tras analizar el líquido sinovial. Hermano con artritis reumatoide."
Diagnósticos:
- artrosis de rodilla (negado: false, antecedente_familiar: false)
- gota (negado: true, antecedente_familiar: false)
- artritis reumatoide (negado: false, antecedente_familiar: true)
"""

plantilla_few_shot = ChatPromptTemplate.from_messages([
    ("system", INSTRUCCIONES_EXTRACCION + "\n" + EJEMPLOS + "\n{instrucciones_formato}"),
    ("human", "Informe:\n{informe}"),
]).partial(instrucciones_formato=parser.get_format_instructions())

cadena_few_shot = plantilla_few_shot | llm | parser
predicciones_few_shot = cadena_few_shot.batch(entradas, config={"max_concurrency": 4}, return_exceptions=True)

tabla_few_shot = evaluar(predicciones_few_shot)

def resumen(tabla):
    return tabla["resultado"].value_counts()

comparacion = pd.DataFrame({"zero-shot": resumen(tabla_zero_shot), "few-shot": resumen(tabla_few_shot)}).fillna(0).astype(int)
display(comparacion)

display(tabla_few_shot[tabla_few_shot["resultado"] != "✅ correcto"])

Diagnósticos correctos: 25 de 31 (81 %)
  ⚠️ atributo erróneo: 3
  ❌ omitido: 3
  ➕ sobrante: 0


,zero-shot,few-shot
resultado,,
⚠️ atributo erróneo,2,3
✅ correcto,26,25
❌ omitido,3,3


,informe,referencia,neg_ref,fam_ref,prediccion,neg_pred,fam_pred,resultado
3,1,cáncer de colon,False,True,cáncer de colon,True,True,⚠️ atributo erróneo
8,3,cáncer de mama,False,True,cáncer de mama,True,True,⚠️ atributo erróneo
16,6,infección cutánea,True,False,,None,None,❌ omitido
20,7,enfermedad cardiovascular,True,True,,None,None,❌ omitido
29,10,migraña sin aura,False,False,migraña sin aura,False,True,⚠️ atributo erróneo
30,10,migraña,False,True,,None,None,❌ omitido


In [ ]:
EJEMPLOS = """
Ejemplos resueltos:

Informe: "Paciente de 60 años con artrosis de rodilla. Se descarta gota tras analizar el líquido sinovial. Hermano con artritis reumatoide."
Diagnósticos:
- artrosis de rodilla (negado: false, antecedente_familiar: false)
- gota (negado: true, antecedente_familiar: false)
- artritis reumatoide (negado: false, antecedente_familiar: true)
"""

plantilla_few_shot = ChatPromptTemplate.from_messages([
    ("system", INSTRUCCIONES_EXTRACCION + "\n" + EJEMPLOS + "\n{instrucciones_formato}"),
    ("human", "Informe:\n{informe}"),
]).partial(instrucciones_formato=parser.get_format_instructions())

cadena_few_shot = plantilla_few_shot | llm | parser
predicciones_few_shot = cadena_few_shot.batch(entradas, config={"max_concurrency": 4}, return_exceptions=True)

tabla_few_shot = evaluar(predicciones_few_shot)

display(tabla_zero_shot[tabla_zero_shot["informe"] == 7])
display(tabla_few_shot[tabla_few_shot["informe"] == 7])

Diagnósticos correctos: 25 de 31 (81 %)
  ⚠️ atributo erróneo: 2
  ❌ omitido: 4
  ➕ sobrante: 0


,informe,referencia,neg_ref,fam_ref,prediccion,neg_pred,fam_pred,resultado
17,7,EPOC,False,False,EPOC,False,False,✅ correcto
18,7,obesidad,False,False,obesidad,False,False,✅ correcto
19,7,síndrome de apnea obstructiva del sueño,False,False,síndrome de apnea obstructiva del sueño,False,False,✅ correcto
20,7,enfermedad cardiovascular,True,True,,None,None,❌ omitido


,informe,referencia,neg_ref,fam_ref,prediccion,neg_pred,fam_pred,resultado
17,7,EPOC,False,False,EPOC,False,False,✅ correcto
18,7,obesidad,False,False,obesidad,False,False,✅ correcto
19,7,síndrome de apnea obstructiva del sueño,False,False,síndrome de apnea obstructiva del sueño,False,False,✅ correcto
20,7,enfermedad cardiovascular,True,True,,None,None,❌ omitido


In [ ]:
resultado = llm_estructurado.invoke(plantilla_extraccion.invoke({"informe": INFORMES[6]["texto"]}))
display(pd.DataFrame([d.model_dump() for d in resultado.diagnosticos]))

print(list(tabla_zero_shot.columns))

,diagnostico,negado,antecedente_familiar,sospecha
0,EPOC,False,False,False
1,obesidad,False,False,False
2,síndrome de apnea obstructiva del sueño,False,False,True


['informe', 'referencia', 'neg_ref', 'fam_ref', 'prediccion', 'neg_pred', 'fam_pred', 'resultado']


---
## Del notebook a Langflow

En la próxima sesión construiremos estos mismos pasos con componentes visuales. Esta es la correspondencia:

| En este notebook | En Langflow (sesión 2) |
|---|---|
| `HumanMessage` / entrada de texto | *Chat Input* |
| `ChatPromptTemplate` con `{variables}` | *Prompt Template* |
| `init_chat_model(..., temperature=...)` | *Language Model* (con su parámetro de temperatura) |
| `with_structured_output` + esquema Pydantic | Componente de salida estructurada |
| `prompt \| llm \| parser` | Componentes unidos por conexiones en el lienzo |
| `.batch()` + función `evaluar()` | — (fíjate en si lo echas de menos) |

---


## Entregable

Termina los ejercicios ✏️ que no te haya dado tiempo a hacer en clase y responde brevemente (3–6 líneas cada una) a las siguientes preguntas en la celda de abajo. Entrega el notebook **ejecutado**, con las salidas visibles.

1. **Ejemplos en el prompt.** ¿Qué cambió al pasar de *zero-shot* a *few-shot*? Indica en qué informes y en qué tipo de error se nota la diferencia.
2. **Tipos de error.** ¿Qué tipo de error comete más el modelo: omisiones, atributos erróneos (negación o antecedente familiar) o diagnósticos sobrantes? ¿Cuál te parece más grave en un contexto clínico y por qué?
3. **Temperatura.** ¿Qué temperatura elegirías para la extracción de diagnósticos y cuál para redactar información para pacientes? ¿Garantiza la temperatura 0 que el resultado sea siempre el mismo?
4. **Incertidumbre.** El informe 7 contiene una *sospecha* de apnea del sueño. ¿Cómo la ha representado el modelo con el esquema original? Propón una modificación del esquema y justifícala.
5. **Evaluación.** Nuestra evaluación usa 10 informes y compara nombres de forma aproximada. Señala dos limitaciones de este método y explica qué harías antes de usar un sistema así con informes reales de un hospital (piensa también en la protección de datos).

### Respuestas

1.Al pasar de zero-shot a few-shot el modelo no mejoró, sino que empeoró ligeramente: de 26 a 24-25 diagnósticos correctos de 31. Aparecieron nuevos errores en el informe 1 (cáncer de colon, atributo erróneo) y en el informe 4 (apendicitis aguda, omitido). Los errores se mantuvieron en los informes 3, 6, 7 y 10. Con solo 10 informes, una diferencia de 1-2 aciertos es muy pequeña y puede deberse al azar, y los ejemplos añadidos no cubrían bien los casos difíciles

2.Los errores más frecuentes en mi evaluación son las omisiones (3-4) y los atributos erróneos (2-3), y no hubo diagnósticos sobrantes. Dentro de lo que cabe la omisión de todo los fallos no es el peor fallo, me parece más grave el atributo erróneo, ya que puede llevar a un tratamiento erroneo.

3.Para extraer diagnósticos elegiría temperatura 0, porque se busca precisión y resultados estables y repetibles. Para redactar información para pacientes se puede usar una temperatura algo mayor (0,5-0,7) para que el texto sea más natural, siempre con revisión, sin llegar a valores altos donde aparecen invenciones o texto sin sentido. Temperatura 0 no garantiza que el resultado sea siempre el mismo: en mi prueba del título con T=0 obtuve 2 respuestas distintas de 5.

4.Con el esquema original el modelo representó la sospecha de apnea del sueño como negado=True, mezclando “no confirmado” con “descartado”. Es un error, porque el informe solo dice que se solicita una polisomnografía por sospecha. Mi modificación fue añadir un campo sospecha: booleano (“True si el diagnóstico no está confirmado”), con el que el modelo marcó negado=False y sospecha=True.

5.La primera limitación sería que solo hay 10 informes sintéticos y 31 diagnósticos, así que los porcentajes tienen mucho margen de error y no se pueden extraer conclusiones sólidas; La segunda la comparación de nombres es aproximada (similitud de texto y sinónimos), por lo que puede dar por buenas coincidencias incorrectas o fallar con sinónimos no previstos, a veces palabras similares en medicina implican mucha diferncia médica, y tampoco evalúa el campo sospecha. Antes de usarlo con informes reales haría falta validarlo con un conjunto mucho mayor anotado por varios profesionales, medir precisión y exhaustividad, y mantener revisión humana. Además, cumplir el RGPD: los datos de salud son categoría especial, así que habría que anonimizar o seudonimizar los informes, tener base legal y contrato con el proveedor, y preferir un modelo desplegado en local o en el entorno del hospital en lugar de enviar datos a una API externa.